In [10]:
import csv
import json
import re
import urllib.parse
import urllib.request
import uuid


def segment_myanmar_text(text):
    """
    မလိမ္မာသော သို့မဟုတ် ပူးနေသော စာသားများအကြား Space များကို
    သပ်ရပ်စွာ ရှင်းလင်းပေးသည့် အခြေခံ Helper Function
    """
    cleaned = re.sub(r"\s+", " ", text).strip()
    return cleaned


def scrape_until_300_sentences():
    # Wikipedia Titles
    seed_titles = [
        "ဦးသန့်",
        "ကုလသမဂ္ဂ",
        "ကုလသမဂ္ဂ အထွေထွေအတွင်းရေးမှူးချုပ်",
        "ကျူးဘား ဒုံးပျံ အကျပ်အတည်း",
        "မြန်မာနိုင်ငံ၏ နိုင်ငံခြားရေးမူဝါဒ",
        "မြန်မာ့သမိုင်း",
        "မြန်မာနိုင်ငံ",
        "ရန်ကုန်မြို့",
        "မန္တလေးမြို့",
        "Bagan",
        "ကုန်းဘောင်ခေတ်",
    ]

    sentences = []
    seen = set()

    print("Wikipedia မှ စာကြောင်းများ စတင် ဆွဲထုတ်နေပါသည်...")

    for title in seed_titles:
        if len(sentences) >= 300:
            break

        encoded_title = urllib.parse.quote(title.strip())
        api_url = f"https://my.wikipedia.org/w/api.php?action=query&prop=extracts&explaintext=1&titles={encoded_title}&format=json"

        # Construct human-readable Wikipedia article link
        source_url = f"https://my.wikipedia.org/wiki/{encoded_title}"

        req = urllib.request.Request(
            api_url, headers={"User-Agent": "Mozilla/5.0"}
        )

        try:
            with urllib.request.urlopen(req) as response:
                data = json.loads(response.read().decode("utf-8"))
                pages = data["query"]["pages"]
                page_id = list(pages.keys())[0]

                if page_id == "-1":
                    continue

                extract_text = pages[page_id].get("extract", "")


                raw_lines = re.split(r"[။?\n]", extract_text)

                for line in raw_lines:
                    cleaned = segment_myanmar_text(line)

                    if (
                        len(cleaned) > 15
                        and cleaned not in seen
                        and not cleaned.startswith("=")
                    ):
                        seen.add(cleaned)
                        # Store sentence paired with its Wikipedia source URL
                        sentences.append((cleaned + " ။", source_url))

                    if len(sentences) >= 300:
                        break
        except Exception as e:
            print(f"Error fetching {title}: {e}")
            continue

    final_sentences = sentences[:300]

    # POLAR Benchmark Format CSV Output File
    output_filename = "formal_text_300_sentences.csv"

    # Header Columns including Source Link
    headers = [
        "ID",
        "Text",
        "Task 1",
        "Political",
        "Racial",
        "Religious",
        "Gender",
        "Other",
        "Keywords",
        "Source Link",
    ]

    with open(
        output_filename, mode="w", newline="", encoding="utf-8-sig"
    ) as file:
        writer = csv.writer(file)
        writer.writerow(headers)

        for sentence, source_link in final_sentences:
            # POLAR Unique ID Generator
            unique_id = f"mya_{uuid.uuid4().hex[:8]}"

            writer.writerow([
                unique_id,    # ID
                sentence,     # Text
                0,            # Task 1 (0 = Non-Polarized)
                0,            # Political
                0,            # Racial
                0,            # Religious
                0,            # Gender
                0,            # Other
                "NULL",       # Keywords
                source_link   # Source Link
            ])

    print("--------------------------------------------------")
    print(f"Total Sentences Processed: {len(final_sentences)}")
    print(f"CSV File Created: {output_filename}")
    print("--------------------------------------------------")


# Run Script
scrape_until_300_sentences()

Wikipedia မှ စာကြောင်းများ စတင် ဆွဲထုတ်နေပါသည်...
--------------------------------------------------
Total Sentences Processed: 300
CSV File Created: formal_text_300_sentences.csv
--------------------------------------------------


In [11]:
import uuid
import pandas as pd
from datasets import load_dataset

# 1.From Hugging Face datasets loading
print("Loading dataset from Hugging Face...")
ds = load_dataset("simbolo-ai/burmese-hatespeech")


df_raw = pd.DataFrame(ds['train'])

# 2. only 300 sentences
df_sample = df_raw.sample(n=300, random_state=42).reset_index(drop=True)

# 3. Simple Word Segmentation Function
def segment_text(text):
    if not isinstance(text, str):
        return ""
    return " ".join(text.split())

# Hugging Face Dataset direct page URL
HF_DATASET_URL = "https://huggingface.co/datasets/simbolo-ai/burmese-hatespeech"

# 4. POLAR Benchmark CSV Format
social_rows = []

for idx, row in df_sample.iterrows():

    text_content = row.get('text', row.get('sentence', ''))

    unique_id = f"mya_{uuid.uuid4().hex[:8]}"
    segmented_text = segment_text(text_content)


    original_label = row.get('label', 0)
    task1_val = 1 if original_label in [1, 'hate', 'hate_speech'] else 0

    source_link = row.get('source_link', row.get('url', HF_DATASET_URL))

    keyphrase_val = row.get('keyphrase', "NULL" if task1_val == 0 else "TBD")

    social_rows.append({
        "ID": unique_id,
        "Text": segmented_text,
        "Task 1": task1_val,
        "Political": 0,
        "Racial": 0,
        "Religious": 0,
        "Gender": 0,
        "Other": 1 if task1_val == 1 else 0,
        "Keywords": "NULL" if task1_val == 0 else "TBD",
        "Keyphrase": keyphrase_val,
        "Source Link": source_link
    })

df_social = pd.DataFrame(social_rows)
csv_filename = "social_media_simbolo_300.csv"
df_social.to_csv(csv_filename, index=False, encoding="utf-8-sig")

print(f"✅ Successfully converted 300 comments and saved to '{csv_filename}'")

Loading dataset from Hugging Face...
✅ Successfully converted 300 comments and saved to 'social_media_simbolo_300.csv'


In [12]:
import os
import re
import pandas as pd

# ၁။ Input / Output Paths
RAW_FILES = [
    "/content/formal_text_300_sentences.csv",
    "/content/social_media_simbolo_300.csv",
]

OUTPUT_DIR = "/content/POLAR_Final_Submission"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ၂။ CSV
df_list = []
for file_path in RAW_FILES:
    if os.path.exists(file_path):
        temp_df = pd.read_csv(file_path)
        df_list.append(temp_df)
    else:
        print(f" Warning: File မတွေ့ပါ -> {file_path}")

if not df_list:
    raise FileNotFoundError(
        "CSV ဖိုင်များ ရှာမတွေ့ပါ။ Upload တင်ထားသော ဖိုင်နာမည်များကို ပြန်စစ်ပါ။"
    )

df = pd.concat(df_list, ignore_index=True)

# Ensure Source Link and Keyphrase columns exist in combined DataFrame
if "Source Link" not in df.columns:
    df["Source Link"] = "NULL"

if "Keyphrase" not in df.columns:
    df["Keyphrase"] = "NULL"

# Task 1 Export: Scraped Raw Data
task1_path = os.path.join(OUTPUT_DIR, "Task1_Raw_Scraped_Data.csv")
df.to_csv(task1_path, index=False, encoding="utf-8-sig")
print(f" Task 1 Exported: {task1_path} ({len(df)} rows)")

# ၃။ Word Segmentation
target_col = next(
    (col for col in ["Text", "comment_text", "comment", "text"] if col in df.columns),
    df.columns[1],
)

pattern = r"(?<![္က-သနှမှဠဧဩဪဿ၌ါ-ှင-ှ])(?=[က-အဧဩဪဿ၌ါ-ှင-ှ])"


def segment_text(text):
    if pd.isna(text):
        return ""
    segmented = re.sub(pattern, " ", str(text))
    return " ".join(segmented.split())


df["segmented_text"] = df[target_col].apply(segment_text)

# Task 2 Export: Word Segmented Data (with Keyphrase & Source Link)
df_task2 = pd.DataFrame({
    "id": df.index + 1,
    "segmented_text": df["segmented_text"],
    "Keyphrase": df["Keyphrase"],
    "Source Link": df["Source Link"],
})
task2_path = os.path.join(OUTPUT_DIR, "Task2_Word_Segmented_Data.csv")
df_task2.to_csv(task2_path, index=False, encoding="utf-8-sig")
print(f" Task 2 Exported: {task2_path} ({len(df_task2)} rows)")

# ၄။ POLAR Criteria Labeling & Keywords/Keyphrase Processing
TOXIC_KEYWORDS = ["စောက်ရူး", "စောက်ပို", "ခွေး", "ရွံစရာ", "ရူး", "အကောင်"]
SEVERE_KEYWORDS = ["သေလိုက်", "သတ်မယ်", "မျိုးဖြုတ်"]
IDENTITY_KEYWORDS = ["ကုလား", "တရုတ်", "ဘင်္ဂါလီ", "ခရစ်ယာန်"]


def label_polar(row):
    text = row["segmented_text"]
    if pd.isna(text) or not str(text).strip():
        return pd.Series([0, 0, 0, 0, 0, 0, "NULL", "NULL"])

    text_str = str(text)
    extracted_keywords = [
        kw
        for kw in (TOXIC_KEYWORDS + SEVERE_KEYWORDS + IDENTITY_KEYWORDS)
        if kw in text_str
    ]

    has_toxic = any(kw in text_str for kw in TOXIC_KEYWORDS)
    has_severe = any(kw in text_str for kw in SEVERE_KEYWORDS)
    has_identity = any(kw in text_str for kw in IDENTITY_KEYWORDS)

    toxicity = 1 if (has_toxic or has_severe) else 0
    severe_toxicity = 1 if has_severe else 0
    insult = 1 if has_toxic else 0
    identity_attack = 1 if has_identity else 0
    insult_toxicity = 1 if (insult == 1 and toxicity == 1) else 0
    severe_insult_toxicity = (
        1 if (severe_toxicity == 1 and insult_toxicity == 1) else 0
    )

    kw_str = (
        "|".join(dict.fromkeys(extracted_keywords))
        if extracted_keywords
        else "NULL"
    )

    # Use existing Keyphrase if present, otherwise default to extracted keywords
    existing_keyphrase = row.get("Keyphrase", "NULL")
    keyphrase_str = (
        existing_keyphrase
        if (pd.notna(existing_keyphrase) and str(existing_keyphrase) != "NULL")
        else kw_str
    )

    return pd.Series([
        toxicity,
        severe_toxicity,
        insult,
        identity_attack,
        insult_toxicity,
        severe_insult_toxicity,
        kw_str,
        keyphrase_str,
    ])


label_cols = [
    "Toxicity",
    "Severe Toxicity",
    "Insult",
    "Identity Attack",
    "Insult Toxicity",
    "Severe Insult Toxicity",
    "Keywords",
    "Keyphrase",
]

df[label_cols] = df.apply(label_polar, axis=1)

# Task 3 Export: Fully Labelled POLAR Data (Includes all metadata, Keyphrase & Source Link)
task3_path = os.path.join(OUTPUT_DIR, "Task3_Fully_Labelled_POLAR_Data.csv")
df.to_csv(task3_path, index=False, encoding="utf-8-sig")
print(f" Task 3 Exported: {task3_path} ({len(df)} rows)")

print("\n--------------------------------------------------")
print(
    " All tasks completed successfully! Check POLAR_Final_Submission folder."
)

 Task 1 Exported: /content/POLAR_Final_Submission/Task1_Raw_Scraped_Data.csv (600 rows)
 Task 2 Exported: /content/POLAR_Final_Submission/Task2_Word_Segmented_Data.csv (600 rows)
 Task 3 Exported: /content/POLAR_Final_Submission/Task3_Fully_Labelled_POLAR_Data.csv (600 rows)

--------------------------------------------------
 All tasks completed successfully! Check POLAR_Final_Submission folder.
